# Task 5 — High-dimensional Kolmogorov experiment
Compare neural Feynman–Kac approximation for arithmetic-basket and max-call payoffs across dimensions 1, 2, 5, 10, 20, and 50. The emphasis is accuracy and amortization rather than claiming universal neural superiority.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.neural_solver import train_neural_feynman_kac, predict_prices, qmc_reference_prices, error_metrics


In [ ]:
# The full reference experiment is already scripted in run_experiments.py.
# Here we demonstrate one 20D arithmetic-basket solve interactively.
d=20
tr=train_neural_feynman_kac(dim=d,payoff='basket',low=80,high=120,strike=100,r=0.05,sigma=0.2,T=1.0,steps=1100,batch_size=1536,antithetic=True,seed=500)
rng=np.random.default_rng(1020)
x=rng.uniform(80,120,size=(64,d))
ref=qmc_reference_prices(x,payoff='basket',strike=100,r=0.05,sigma=0.2,T=1.0,n_paths=4096,seed=900)
pred=predict_prices(tr.model,x,100)
error_metrics(pred,ref)


In [ ]:
results=pd.read_csv(ROOT/'results'/'task5_dimension_scaling.csv')
results


In [ ]:
plt.figure(figsize=(6,4))
for payoff,g in results.groupby('payoff'):
    plt.plot(g.dimension,100*g.relative_l2,'o-',label=payoff)
plt.xlabel('dimension d'); plt.ylabel('relative L2 error (%)'); plt.legend(); plt.grid(True,alpha=.25);


The arithmetic basket benefits from averaging/concentration, so a max-call benchmark is included as a less forgiving geometry. The experiment does not prove dimension-independent complexity; it demonstrates useful simulation-and-regression accuracy in dimensions where tensor-product finite-difference grids are impractical.